**Machine Learning Analysis**

**student**: Bera Acar(33834)
**course**: DSA210
**Term**: Spring 25-26


**SETUP**

In [18]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier, GradientBoostingClassifier
from sklearn.cluster import KMeans
from sklearn.metrics import (mean_absolute_error, r2_score, mean_squared_error,
                             classification_report, confusion_matrix, silhouette_score)
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from sklearn.pipeline import Pipeline

**LOAD and CLEAN DATA**


In [6]:
df = pd.read_csv("NBA_Advanced_Salary_Data.csv")

df = df.dropna(subset=["Salary", "PER", "WS", "BPM"])
df = df[df["Salary"] > 0]
df = df[df["G"] >= 20]

FEATURES = ["Age", "G", "MP", "PER", "TS%", "TRB%", "AST%", "STL%",
            "BLK%", "USG%", "OWS", "DWS", "WS", "WS/48", "BPM", "VORP"]

df_model = df[FEATURES + ["Salary", "Player_Clean", "Season"]].dropna()

print(f"✅ Cleaned Data: {df_model.shape[0]} players, {len(FEATURES)} features")
print(f"   Seasons: {df_model['Season'].unique()}")
df_model["LogSalary"] = np.log1p(df_model["Salary"])

X = df_model[FEATURES]
y_log = df_model["LogSalary"]
y_raw = df_model["Salary"]

✅ Cleaned Data: 683 players, 16 features
   Seasons: ['2022-23' '2023-24']


**REGRESSION**

In [7]:
print("\n" + "="*55)
print("TASK 1: REGRESSION — Salary Prediction")
print("="*55)

X_train, X_test, y_train, y_test = train_test_split(
    X, y_log, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

models = {
    "Linear Regression": LinearRegression(),
    "Ridge (α=10)":      Ridge(alpha=10),
    "Lasso (α=0.01)":    Lasso(alpha=0.01, max_iter=5000),
    "Random Forest":     RandomForestRegressor(n_estimators=200, random_state=42)
}

results = {}
for name, model in models.items():
    if name == "Random Forest":
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)
        cv_r2  = cross_val_score(model, X, y_log, cv=5, scoring="r2").mean()
    else:
        model.fit(X_train_sc, y_train)
        y_pred = model.predict(X_test_sc)
        cv_r2  = cross_val_score(model, scaler.transform(X), y_log, cv=5, scoring="r2").mean()

    r2  = r2_score(y_test, y_pred)
    mae = mean_absolute_error(np.expm1(y_test), np.expm1(y_pred)) / 1e6  # million $

    results[name] = {"R²": r2, "CV R²": cv_r2, "MAE ($M)": mae}
    print(f"  {name:<25} R²={r2:.3f}  CV-R²={cv_r2:.3f}  MAE=${mae:.2f}M")


best_model = models["Random Forest"]

# Feature importance
importances = pd.Series(best_model.feature_importances_, index=FEATURES).sort_values(ascending=False)


TASK 1: REGRESSION — Salary Prediction
  Linear Regression         R²=0.659  CV-R²=0.622  MAE=$4.52M
  Ridge (α=10)              R²=0.659  CV-R²=0.622  MAE=$4.51M
  Lasso (α=0.01)            R²=0.654  CV-R²=0.620  MAE=$4.60M
  Random Forest             R²=0.631  CV-R²=0.630  MAE=$4.77M


**Note:** In terms of test R², Ridge Regression achieved the best result (0.659). However, when cross-validation scores are compared, the difference between models is small. Random Forest was preferred due to its ability to provide feature importance analysis, making the results more interpretable.

**MODEL COMPARISON**

In [9]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle("Task 1: Regression — Maaş Tahmini", fontsize=14, fontweight="bold")

# 1a) R² Comparison
model_names = list(results.keys())
r2_vals  = [results[m]["R²"]     for m in model_names]
cvr2_vals= [results[m]["CV R²"]  for m in model_names]
x_pos = np.arange(len(model_names))
axes[0].bar(x_pos - 0.2, r2_vals,  0.35, label="Test R²",  color="#4C72B0")
axes[0].bar(x_pos + 0.2, cvr2_vals,0.35, label="CV R²",    color="#DD8452")
axes[0].set_xticks(x_pos)
axes[0].set_xticklabels(model_names, rotation=15, ha="right", fontsize=8)
axes[0].set_ylabel("R² Score")
axes[0].set_title("Model Karşılaştırma")
axes[0].legend()
axes[0].set_ylim(0, 1)

# 1b) Actual vs Predicted (Random Forest)
rf = models["Random Forest"]
y_pred_all = rf.predict(X)
axes[1].scatter(np.expm1(y_log)/1e6, np.expm1(y_pred_all)/1e6,
                alpha=0.4, s=20, color="#4C72B0")
lims = [0, 55]
axes[1].plot(lims, lims, "r--", lw=1.5, label="Mükemmel tahmin")
axes[1].set_xlabel("Gerçek Maaş ($M)")
axes[1].set_ylabel("Tahmin Edilen Maaş ($M)")
axes[1].set_title("Random Forest: Actual vs Predicted")
axes[1].legend()

# 1c) Feature importance (top 10)
top10 = importances.head(10)
axes[2].barh(top10.index[::-1], top10.values[::-1], color="#4C72B0")
axes[2].set_title("Feature Importance (Top 10)")
axes[2].set_xlabel("Importance")

plt.tight_layout()
plt.savefig("regression.png", dpi=150, bbox_inches="tight")
plt.close()


**Clustering**

In [14]:
print("\n" + "="*55)
print("TASK 2: CLUSTERING")
print("="*55)

CLUSTER_FEATURES = ["PER", "AST%", "TRB%", "BLK%", "STL%", "USG%", "WS/48", "OBPM" if "OBPM" in df_model.columns else "BPM"]
CLUSTER_FEATURES = [f for f in CLUSTER_FEATURES if f in df_model.columns]

X_clust = df_model[CLUSTER_FEATURES].dropna()
scaler_c = StandardScaler()
X_clust_sc = scaler_c.fit_transform(X_clust)

# Elbow + Silhouette Method
inertias, sil_scores = [], []
K_range = range(2, 9)
for k in K_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(X_clust_sc)
    inertias.append(km.inertia_)
    sil_scores.append(silhouette_score(X_clust_sc, labels))

best_k = K_range[np.argmax(sil_scores)]
print(f"  Best K (silhouette): {best_k}  →  score={max(sil_scores):.3f}")

# Final clustering
km_final = KMeans(n_clusters=best_k, random_state=42, n_init=10)
df_model = df_model.loc[X_clust.index].copy()
df_model["Cluster"] = km_final.fit_predict(X_clust_sc)

# Cluster Profiles
cluster_profile = df_model.groupby("Cluster")[CLUSTER_FEATURES + ["Salary"]].mean()
print("\n  Cluster Profiles (average):")
print(cluster_profile.round(2).to_string())


def label_cluster(row):
    if row["AST%"] > 22:    return "Playmaker"
    elif row["BLK%"] > 3.5: return "Rim Protector"
    elif row["USG%"] > 24:  return "Scorer"
    elif row["TRB%"] > 18:  return "Rebounder"
    else:                   return "Role Player"

cluster_profile["Role"] = cluster_profile.apply(label_cluster, axis=1)
role_map = cluster_profile["Role"].to_dict()
df_model["Role"] = df_model["Cluster"].map(role_map)
print("\n  Cluster -> Role Labels:")
for c, r in role_map.items():
    n = (df_model["Cluster"] == c).sum()
    avg_sal = df_model[df_model["Cluster"]==c]["Salary"].mean()/1e6
    print(f"    Cluster {c}: {r:<16} ({n} players, average salary=${avg_sal:.1f}M)")


TASK 2: CLUSTERING
  Best K (silhouette): 4  →  score=0.301

  Cluster Profiles (average):
           PER   AST%   TRB%  BLK%  STL%   USG%  WS/48   BPM       Salary
Cluster                                                                  
0        18.72  24.87   9.11  1.32  1.56  25.91   0.13  2.27  22105514.74
1        11.24  12.01   8.05  1.27  1.55  16.37  -0.43 -2.01   6861456.06
2        16.64   8.57  16.47  4.13  1.30  16.59   0.14 -0.13   8574032.94
3         4.65  15.07   6.13  0.75  1.43  17.80 -72.67 -7.73   2756881.33

  Cluster -> Role Labels:
    Cluster 0: Playmaker        (163 players, average salary=$22.1M)
    Cluster 1: Role Player      (374 players, average salary=$6.9M)
    Cluster 2: Rim Protector    (140 players, average salary=$8.6M)
    Cluster 3: Role Player      (6 players, average salary=$2.8M)


**Cluster Plotting**

In [15]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle("Task 2: K-Means Clustering — Player Archetypes", fontsize=14, fontweight="bold")

colors = plt.cm.Set2(np.linspace(0, 1, best_k))

# 2a) Elbow + Silhouette
ax2 = axes[0].twinx()
axes[0].plot(list(K_range), inertias, "b-o", label="Inertia")
ax2.plot(list(K_range), sil_scores, "r-s", label="Silhouette")
axes[0].set_xlabel("K")
axes[0].set_ylabel("Inertia", color="b")
ax2.set_ylabel("Silhouette Score", color="r")
axes[0].set_title("Elbow & Silhouette")
axes[0].axvline(best_k, color="gray", linestyle="--", alpha=0.7)

# 2b) Scatter: USG% vs WS/48
for i, (cid, role) in enumerate(role_map.items()):
    mask = df_model["Cluster"] == cid
    axes[1].scatter(df_model.loc[mask, "USG%"], df_model.loc[mask, "WS/48"],
                    alpha=0.6, s=30, label=role, color=colors[i])
axes[1].set_xlabel("USG% (Usage Rate)")
axes[1].set_ylabel("WS/48 (Win Shares per 48)")
axes[1].set_title("Cluster: USG% vs WS/48")
axes[1].legend(fontsize=7)

# 2c) Average salary per cluster
roles = [role_map[c] for c in sorted(role_map)]
salaries = [df_model[df_model["Cluster"]==c]["Salary"].mean()/1e6 for c in sorted(role_map)]
bar_colors = [colors[c] for c in sorted(role_map)]
bars = axes[2].bar(roles, salaries, color=bar_colors)
axes[2].set_ylabel("Average Salary ($M)")
axes[2].set_title("Average Salary per Cluster")
axes[2].tick_params(axis="x", rotation=20)
for bar, sal in zip(bars, salaries):
    axes[2].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.2,
                 f"${sal:.1f}M", ha="center", va="bottom", fontsize=8)

plt.tight_layout()
plt.savefig("clustering.png", dpi=150, bbox_inches="tight")
plt.close()

**CLASSIFICATION**

In [19]:
print("\n" + "="*55)
print("TASK 3: CLASSIFICATION — Overpaid / Underpaid")
print("="*55)

# Use regression model to predict salary, then compare with actual
df_model["PredSalary"] = np.expm1(best_model.predict(df_model[FEATURES]))

# SalaryRatio = Actual / Predicted
df_model["SalaryRatio"] = df_model["Salary"] / df_model["PredSalary"]

# Thresholds determined by 33rd and 67th percentiles
# to ensure balanced class representation
q33 = df_model["SalaryRatio"].quantile(0.33)
q67 = df_model["SalaryRatio"].quantile(0.67)
print(f"  Thresholds: Underpaid < {q33:.2f} < Fair < {q67:.2f} < Overpaid")

def assign_label(ratio):
    if ratio > q67:    return "Overpaid"
    elif ratio < q33:  return "Underpaid"
    else:              return "Fair"

df_model["PayLabel"] = df_model["SalaryRatio"].apply(assign_label)

print("\n  Class Distribution:")
print(df_model["PayLabel"].value_counts())

# Classification
X_clf = df_model[FEATURES]
y_clf = df_model["PayLabel"]

X_tr, X_te, y_tr, y_te = train_test_split(
    X_clf, y_clf, test_size=0.2, random_state=42, stratify=y_clf)

# Model 1: Random Forest Classifier
rf_clf = RandomForestClassifier(
    n_estimators=200,
    random_state=42,
    class_weight="balanced"  # handles class imbalance
)
rf_clf.fit(X_tr, y_tr)
y_pred_rf = rf_clf.predict(X_te)

# Model 2: Gradient Boosting Classifier
gb_clf = GradientBoostingClassifier(
    n_estimators=200,
    learning_rate=0.05,
    random_state=42
)
gb_clf.fit(X_tr, y_tr)
y_pred_gb = gb_clf.predict(X_te)

cv_rf = cross_val_score(rf_clf, X_clf, y_clf, cv=5, scoring="accuracy").mean()
cv_gb = cross_val_score(gb_clf, X_clf, y_clf, cv=5, scoring="accuracy").mean()

print(f"\n  Random Forest:     Accuracy={accuracy_score(y_te, y_pred_rf):.3f}  CV={cv_rf:.3f}")
print(classification_report(y_te, y_pred_rf))

print(f"\n  Gradient Boosting: Accuracy={accuracy_score(y_te, y_pred_gb):.3f}  CV={cv_gb:.3f}")
print(classification_report(y_te, y_pred_gb))

# Gradient Boosting performed better
best_clf    = gb_clf
y_pred_best = y_pred_gb



TASK 3: CLASSIFICATION — Overpaid / Underpaid
  Thresholds: Underpaid < 0.91 < Fair < 1.12 < Overpaid

  Class Distribution:
PayLabel
Fair         231
Underpaid    226
Overpaid     226
Name: count, dtype: int64

  Random Forest:     Accuracy=0.409  CV=0.400
              precision    recall  f1-score   support

        Fair       0.51      0.51      0.51        47
    Overpaid       0.36      0.33      0.34        45
   Underpaid       0.35      0.38      0.37        45

    accuracy                           0.41       137
   macro avg       0.41      0.41      0.41       137
weighted avg       0.41      0.41      0.41       137


  Gradient Boosting: Accuracy=0.365  CV=0.420
              precision    recall  f1-score   support

        Fair       0.52      0.47      0.49        47
    Overpaid       0.28      0.27      0.27        45
   Underpaid       0.31      0.36      0.33        45

    accuracy                           0.36       137
   macro avg       0.37      0.36      0.

In [23]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle("Task 3: Classification — Overpaid / Underpaid / Fair",
             fontsize=13, fontweight="bold")

label_order  = ["Overpaid", "Fair", "Underpaid"]
label_colors = {"Overpaid": "#E74C3C", "Fair": "#2ECC71", "Underpaid": "#3498DB"}

# 3a) Confusion Matrix
cm = confusion_matrix(y_te, y_pred_best, labels=label_order)
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=label_order, yticklabels=label_order, ax=axes[0])
axes[0].set_title("Confusion Matrix\n(Gradient Boosting)")
axes[0].set_xlabel("Predicted")
axes[0].set_ylabel("Actual")

# 3b) Actual vs Predicted Salary (colored by label)
for label in label_order:
    mask = df_model["PayLabel"] == label
    axes[1].scatter(df_model.loc[mask, "PredSalary"] / 1e6,
                    df_model.loc[mask, "Salary"] / 1e6,
                    alpha=0.5, s=20, label=label, color=label_colors[label])
axes[1].plot([0, 55], [0, 55], "k--", lw=1.5)
axes[1].set_xlabel("Predicted Salary ($M)")
axes[1].set_ylabel("Actual Salary ($M)")
axes[1].set_title("Actual vs Predicted (Labeled)")
axes[1].legend()

# 3c) Feature Importance
fi = pd.Series(best_clf.feature_importances_, index=FEATURES)\
       .sort_values(ascending=False).head(10)
axes[2].barh(fi.index[::-1], fi.values[::-1], color="#9B59B6")
axes[2].set_title("Feature Importance\n(Gradient Boosting)")
axes[2].set_xlabel("Importance")

plt.tight_layout()
plt.savefig("classification.png", dpi=150, bbox_inches="tight")
plt.close()
